In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)
## knockout the btuB gene
model0['lb_list']['EX_cbl1_e ']=0
model0['lb_list']['EX_cbl1_e ']=0

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print(distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
            
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
            
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
        
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0]
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        pass
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
            
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
            
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                        
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])
                                
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death        
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                    
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                        
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
            
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
        
        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
        
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
        
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([25, 27,  9, 37, 36, 37, 30, 37, 13, 12, 34, 23, 16, 10, 20, 21, 18,
       32, 25]), 1: array([17, 23,  1, 31, 21, 13, 21,  0, 26, 29,  7, 37, 15, 35, 27, 17, 35,
       17, 16])}
1
strain_number:  0 28.842105263157894
strain_various:  0 11.268321492222237
strain_number:  1 20.63157894736842
strain_various:  1 10.751795572519086
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 34.1578947368421
strain_various:  0 10.022412556994642
strain_number:  1 20.68421052631579
strain_various:  1 4.746773215109257
fd_r_threshold:  [1.05, 383.8573279819437]
slip_r:  77.61146559638874
3
strain_number:  0 40.63157894736842
strain_various:  0 12.372339804732462
strain_number:  1 20.736842105263158
strain_various:  1 5.209728806591294
fd_r_threshold:  [1.05, 383.8573279819437, 9.031188678459452]
slip_r:  79.20770333208063
4
strain_number:  0 48.36842105263158
strain_various:  0 10.464026226808983
strain_number:  1 20.789473684210527
strain_various:  1 5.115291290449605
fd_r_threshol

glc__D_e_1:  -5.846240875371674
glc__D_e_1:  -3.8911150913034764
glc__D_e_1:  -2.641617511804662
glc__D_e_1:  -5.156171858902889
glc__D_e_1:  -6.8128634193592115
glc__D_e_1:  -6.040641310355241
glc__D_e_1:  -2.535080243174808
glc__D_e_1:  -7.016804664936297
glc__D_e_1:  -7.905625960234309
glc__D_e_1:  -4.972937284132013
glc__D_e_1:  -0.28300457306271243
glc__D_e_1:  -1.3221813641634945
glc__D_e_1:  -7.834956459934866
glc__D_e_1:  -4.902267783832571
glc__D_e_1:  -2.7936668455939397
glc__D_e_1:  -6.783598748689613
glc__D_e_1:  -7.617254809323404
glc__D_e_1:  -4.684566133221108
glc__D_e_1:  -2.49442607639254
glc__D_e_1:  -5.992565460822398
glc__D_e_1:  -10.334263852342232
glc__D_e_1:  -8.584478851304164
glc__D_e_1:  -2.6527184766478102
glc__D_e_1:  -4.183687786414408
glc__D_e_1:  -8.207765521115963
glc__D_e_1:  -7.281098990311199
glc__D_e_1:  -6.248949993312268
glc__D_e_1:  -7.8355110568480155
glc__D_e_1:  -6.623935182854908
glc__D_e_1:  -7.85773521914847
glc__D_e_1:  -3.10325547475788
gl

glc__D_e_1:  -0.4919738019244453
glc__D_e_1:  -0.7541680973688063
glc__D_e_1:  -2.605745820865544
glc__D_e_1:  -1.2005442286012826
glc__D_e_1:  -1.8201571213353638
glc__D_e_1:  -0.2775895510568478
glc__D_e_1:  -0.08438015352740358
glc__D_e_1:  -0.5986097801591201
strain_number:  0 2.3157894736842106
strain_various:  0 0.7292845505553168
strain_number:  1 1.7894736842105263
strain_various:  1 0.6942582083301535
fd_r_threshold:  [1.05, 383.8573279819437, 9.031188678459452, 7.581456000040465, 6.44941744457106, 6.665372404060897, 5.421264268000513, 3.4657232667387228, 4.664129808985298, 4.155144320808218, 6.481195267342565, 6.005159723093067, 3.51652304819268, 3.9367793894417273, 3.7394805051650293, 3.7599319727891163, 4.331944444444444]
slip_r:  3.8569318720065993
18
glc__D_e_1:  -0.7212366276073077
glc__D_e_1:  -0.4888066569456888
glc__D_e_1:  -0.3268088277594632
glc__D_e_1:  -0.8877910813593186
glc__D_e_1:  -0.2858787936222251
glc__D_e_1:  -0.7503909322627802
glc__D_e_1:  -0.31419587459

glc__D_e_1:  -0.017828823988962905
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 383.8573279819437, 9.031188678459452, 7.581456000040465, 6.44941744457106, 6.665372404060897, 5.421264268000513, 3.4657232667387228, 4.664129808985298, 4.155144320808218, 6.481195267342565, 6.005159723093067, 3.51652304819268, 3.9367793894417273, 3.7394805051650293, 3.7599319727891163, 4.331944444444444, 3.805555555555556, 6.861111111111111, 4.833333333333333, 5.0, 4.0, 10.0, 1.0, 1.0, 1.0, 1.0, 1.0]
slip_r:  1.0
29
glc__D_e_1:  -0.1262419537113857
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 383.8573279819437, 9.031188678459452, 7.581456000040465, 6.44941744457106, 6.665372404060897, 5.421264268000513, 3.4657232667387228, 4.664129808985298, 4.155144320808218, 6.481195267342565, 6.00

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([25,  0, 23, 19, 34,  4, 34,  3, 14, 34, 11, 37,  6,  8, 25, 21, 17,
       28, 17]), 1: array([ 2,  5, 38, 39, 39, 20, 30, 26, 25, 38, 35,  3, 20,  0, 22, 27,  2,
        0,  2])}
1
strain_number:  0 22.263157894736842
strain_various:  0 13.427552605559558
strain_number:  1 19.94736842105263
strain_various:  1 15.041954255070529
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 26.263157894736842
strain_various:  0 5.701727395347723
strain_number:  1 20.105263157894736
strain_various:  1 11.867224820097912
fd_r_threshold:  [1.05, 601.6081147662944]
slip_r:  121.1616229532589
3
strain_number:  0 31.105263157894736
strain_various:  0 5.398778973474663
strain_number:  1 20.263157894736842
strain_various:  1 10.577245215248169
fd_r_threshold:  [1.05, 601.6081147662944, 14.889054457019302]
slip_r:  123.92943384466273
4
strain_number:  0 36.94736842105263
strain_various:  0 8.641842421799492
strain_number:  1 20.42105263157895
strain_various:  1 9.996398243334205
fd_r_thre

glc__D_e_1:  -5.592418873898148
glc__D_e_1:  -1.8800202385538696
glc__D_e_1:  -6.361744660315359
glc__D_e_1:  -29.541395121496496
glc__D_e_1:  -7.060959280937805
glc__D_e_1:  -9.078014519690399
glc__D_e_1:  -26.235109946101137
glc__D_e_1:  -3.394888900711525
strain_number:  0 15.789473684210526
strain_various:  0 10.045050875501463
strain_number:  1 7.842105263157895
strain_various:  1 3.4376117458543014
fd_r_threshold:  [1.05, 601.6081147662944, 14.889054457019302, 6.807804102642341, 8.642211406631354, 7.54464377132704, 5.84568939790132, 3.8032946926802764, 4.446545013256286, 3.989032141539729, 4.176826904671103, 5.21648405254004]
slip_r:  4.326436560937486
13
glc__D_e_1:  -5.396800544269265
glc__D_e_1:  -7.453719050796131
glc__D_e_1:  -0.42539621457243104
glc__D_e_1:  -0.9727804870073977
glc__D_e_1:  -7.722810091284507
glc__D_e_1:  -7.464817608912254
glc__D_e_1:  -2.230824408992279
glc__D_e_1:  -1.3028311254297997
glc__D_e_1:  -2.4644216746289485
glc__D_e_1:  -4.161554976324893
glc__

glc__D_e_1:  -1.7200137096465191
glc__D_e_1:  -0.6413096627562149
glc__D_e_1:  -0.3309861426098155
glc__D_e_1:  -0.03259886567984016
glc__D_e_1:  -1.1823891199975651
glc__D_e_1:  -0.29284779144604345
glc__D_e_1:  -1.4344223125530582
glc__D_e_1:  -0.4748292570777891
glc__D_e_1:  -0.36935391925512717
glc__D_e_1:  -1.5522575943193546
glc__D_e_1:  -1.6807155075395754
glc__D_e_1:  -0.47904034967382536
glc__D_e_1:  -0.5144420675840349
glc__D_e_1:  -0.5989849028220358
glc__D_e_1:  -0.8751663455459563
glc__D_e_1:  -1.3480348718930126
glc__D_e_1:  -1.1935904500922185
glc__D_e_1:  -0.7254655811629449
glc__D_e_1:  -0.28926481626628053
glc__D_e_1:  -1.8768444787142087
strain_number:  0 2.3684210526315788
strain_various:  0 0.9296590385608259
strain_number:  1 1.5263157894736843
strain_various:  1 0.6781104593013224
fd_r_threshold:  [1.05, 601.6081147662944, 14.889054457019302, 6.807804102642341, 8.642211406631354, 7.54464377132704, 5.84568939790132, 3.8032946926802764, 4.446545013256286, 3.9890321

glc__D_e_1:  -0.06147059868742066
glc__D_e_1:  -0.1036828258934302
glc__D_e_1:  -0.06226033033523487
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 601.6081147662944, 14.889054457019302, 6.807804102642341, 8.642211406631354, 7.54464377132704, 5.84568939790132, 3.8032946926802764, 4.446545013256286, 3.989032141539729, 4.176826904671103, 5.21648405254004, 4.6567464973207455, 4.0026177101536495, 3.9386306556598765, 4.936485260770976, 3.170521541950114, 4.27138888888889, 8.956666666666667, 3.7847222222222223, 3.9722222222222223, 5.361111111111111, 4.75, 2.75, 3.0, 1.0, 3.0, 3.0]
slip_r:  2.55
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 601.6081147662944, 14.889054457019302, 6.807804102642341, 8.642211406631354, 7.54464377132704, 5.84568939790132, 3.8032946926802764, 4.446545013256286, 3.98903

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([29, 26, 13,  7, 21,  8, 33, 25,  7,  2,  5, 35, 23, 31, 31, 34, 24,
       18,  9]), 1: array([17, 11, 39,  6, 15, 21, 38,  6,  4, 19, 30,  8, 25, 34,  7, 11, 27,
       35, 13])}
1
strain_number:  0 23.63157894736842
strain_various:  0 12.942018578079164
strain_number:  1 19.526315789473685
strain_various:  1 11.789238719463082
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.0
strain_various:  0 12.165525060596439
strain_number:  1 19.526315789473685
strain_various:  1 5.143912304366576
fd_r_threshold:  [1.05, 34.2719475812088]
slip_r:  7.694389516241761
3
strain_number:  0 33.26315789473684
strain_various:  0 10.355990411164056
strain_number:  1 19.526315789473685
strain_various:  1 4.044078465747062
fd_r_threshold:  [1.05, 34.2719475812088, 17.84104041081207]
slip_r:  11.052597598404173
4
strain_number:  0 39.578947368421055
strain_various:  0 10.137558044346433
strain_number:  1 19.526315789473685
strain_various:  1 3.8301906102442635
fd_r_threshold:  [1.05,

glc__D_e_1:  -3.8582318326427067
glc__D_e_1:  -7.903755489507533
glc__D_e_1:  -11.586071780611745
glc__D_e_1:  -5.669798067177818
glc__D_e_1:  -2.237035413399494
glc__D_e_1:  -7.210552353826797
glc__D_e_1:  -13.485656795620644
glc__D_e_1:  -6.591820190152617
glc__D_e_1:  -2.901164008982864
glc__D_e_1:  -5.415718356081092
glc__D_e_1:  -13.261870825760493
glc__D_e_1:  -9.865848884255815
strain_number:  0 14.736842105263158
strain_various:  0 5.618035855154903
strain_number:  1 7.684210526315789
strain_various:  1 2.65672199244715
fd_r_threshold:  [1.05, 34.2719475812088, 17.84104041081207, 9.30239336796806, 7.890644533121293, 8.4694055035343, 6.889132892885887, 3.7583383815167277, 3.743315328639129, 4.443334573381672, 5.172502463103298, 3.5024645822908047]
slip_r:  4.123991065786326
13
glc__D_e_1:  -4.181494475722773
glc__D_e_1:  -3.0494871618878805
glc__D_e_1:  -2.13496731816851
glc__D_e_1:  -5.141314183932552
glc__D_e_1:  -10.131237291979474
glc__D_e_1:  -6.375430145643874
glc__D_e_1: 

glc__D_e_1:  -0.533047792992245
glc__D_e_1:  -0.8198107839845736
glc__D_e_1:  -0.4022625230270085
glc__D_e_1:  -0.24781810122621462
glc__D_e_1:  -0.3355102188268406
glc__D_e_1:  -0.3979001122728094
glc__D_e_1:  -0.24345569047201554
glc__D_e_1:  -0.5829159894158302
glc__D_e_1:  -0.37618168877460034
glc__D_e_1:  -0.8904113154063169
glc__D_e_1:  -0.5235922315253141
glc__D_e_1:  -0.45136178265551274
glc__D_e_1:  -0.7595843236033368
glc__D_e_1:  -0.5240903633952957
glc__D_e_1:  -0.32698250366925397
glc__D_e_1:  -1.4919376046614719
glc__D_e_1:  -0.14178942900966796
glc__D_e_1:  -1.318842109683982
glc__D_e_1:  -0.5077451504524915
glc__D_e_1:  -1.0219747770842083
strain_number:  0 2.3157894736842106
strain_various:  0 0.6531407182100452
strain_number:  1 1.4210526315789473
strain_various:  1 0.7480352843974681
fd_r_threshold:  [1.05, 34.2719475812088, 17.84104041081207, 9.30239336796806, 7.890644533121293, 8.4694055035343, 6.889132892885887, 3.7583383815167277, 3.743315328639129, 4.44333457338

glc__D_e_1:  -0.07315768722513927
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 34.2719475812088, 17.84104041081207, 9.30239336796806, 7.890644533121293, 8.4694055035343, 6.889132892885887, 3.7583383815167277, 3.743315328639129, 4.443334573381672, 5.172502463103298, 3.5024645822908047, 5.549773334535924, 4.623513472717726, 3.9648028470647523, 3.482076247165533, 5.872363945578231, 4.562777777777777, 6.638888888888888, 3.25, 5.861111111111111, 4.25, 2.5, 6.0, 3.0, 2.0, 1.0, 1.0, 2.0]
slip_r:  1.8
30
glc__D_e_1:  -0.15368564160715903
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 34.2719475812088, 17.84104041081207, 9.30239336796806, 7.890644533121293, 8.4694055035343, 6.889132892885887, 3.7583383815167277, 3.743315328639129, 4.443334573381672, 5.172502463103298, 

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([37, 22, 27, 35, 15,  2, 17,  9, 15,  6, 24,  4, 22, 25, 37, 25, 19,
       22,  9]), 1: array([36, 10,  4, 31, 29, 16, 36,  6,  9, 29, 24, 27, 34, 11, 12, 12, 29,
       14, 16])}
1
strain_number:  0 23.105263157894736
strain_various:  0 12.417706953515191
strain_number:  1 20.473684210526315
strain_various:  1 10.76235359239072
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 27.36842105263158
strain_various:  0 11.183684198141833
strain_number:  1 20.473684210526315
strain_various:  1 4.827208205388674
fd_r_threshold:  [1.05, 71.43590726094925]
slip_r:  15.12718145218985
3
strain_number:  0 32.421052631578945
strain_various:  0 12.30453827048628
strain_number:  1 20.473684210526315
strain_various:  1 3.7608568599894414
fd_r_threshold:  [1.05, 71.43590726094925, 7.5392657857893735]
slip_r:  16.425034609347726
4
strain_number:  0 38.526315789473685
strain_various:  0 12.583901248703405
strain_number:  1 20.473684210526315
strain_various:  1 3.746836153801618
fd_r_th

glc__D_e_1:  -5.595767645356467
glc__D_e_1:  -9.149498783555476
glc__D_e_1:  -9.730740802276802
glc__D_e_1:  -5.974933655941202
glc__D_e_1:  -1.784390802854306
glc__D_e_1:  -2.331775075289273
glc__D_e_1:  -14.000012232162266
glc__D_e_1:  -6.437501578261731
glc__D_e_1:  -0.8842578484467833
glc__D_e_1:  -4.382397232876642
glc__D_e_1:  -11.66088698296712
glc__D_e_1:  -7.0819613663982155
glc__D_e_1:  -1.8997744383517303
glc__D_e_1:  -5.397913822781589
glc__D_e_1:  -15.12260125764338
glc__D_e_1:  -8.07432023037456
strain_number:  0 14.052631578947368
strain_various:  0 5.462539261191209
strain_number:  1 8.0
strain_various:  1 3.583588322220885
fd_r_threshold:  [1.05, 71.43590726094925, 7.5392657857893735, 7.664237009468243, 7.840695332112287, 12.077744383915622, 8.8651403048488, 5.067616610576539, 5.656699009106944, 4.638876608972549, 4.242751051095104, 4.967541640765254]
slip_r:  4.914696984103278
13
glc__D_e_1:  -3.629300874355888
glc__D_e_1:  -1.8286195120884847
glc__D_e_1:  -0.99055778

glc__D_e_1:  -0.9771445074103369
glc__D_e_1:  -0.5939641815802608
glc__D_e_1:  -0.053091237696360594
glc__D_e_1:  -0.07195063567750593
glc__D_e_1:  -0.5861802623092225
glc__D_e_1:  -0.30952583838894787
glc__D_e_1:  -0.026768564163625053
glc__D_e_1:  -0.5409981907953418
glc__D_e_1:  -0.53142831429838
glc__D_e_1:  -0.7911931756613682
glc__D_e_1:  -0.6367487538605742
glc__D_e_1:  -0.08822293889779065
glc__D_e_1:  -1.3114340404606275
glc__D_e_1:  -1.1569896186598334
glc__D_e_1:  -1.607376575613134
glc__D_e_1:  -1.154718175625149
glc__D_e_1:  -1.6689478022568656
strain_number:  0 2.6842105263157894
strain_various:  0 0.653140718210045
strain_number:  1 1.263157894736842
strain_various:  1 0.5469634129164876
fd_r_threshold:  [1.05, 71.43590726094925, 7.5392657857893735, 7.664237009468243, 7.840695332112287, 12.077744383915622, 8.8651403048488, 5.067616610576539, 5.656699009106944, 4.638876608972549, 4.242751051095104, 4.967541640765254, 4.237197429252704, 3.7737463579302712, 3.61377657866023

strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 71.43590726094925, 7.5392657857893735, 7.664237009468243, 7.840695332112287, 12.077744383915622, 8.8651403048488, 5.067616610576539, 5.656699009106944, 4.638876608972549, 4.242751051095104, 4.967541640765254, 4.237197429252704, 3.7737463579302712, 3.613776578660237, 5.911512975560595, 3.8852777777777785, 2.840277777777778, 7.541666666666667, 3.2222222222222223, 5.083333333333334, 4.75, 6.75, 4.25, 0.0, 3.0, 3.0, 1.0, 0.0]
slip_r:  1.4
30
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 71.43590726094925, 7.5392657857893735, 7.664237009468243, 7.840695332112287, 12.077744383915622, 8.8651403048488, 5.067616610576539, 5.656699009106944, 4.638876608972549, 4.242751051095104, 4.967541640765254, 4.237197429252704, 3.7737463579302712, 3.613776578660237, 5.9

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 0, 12, 20,  1, 29, 38, 19, 28,  9, 10, 31, 12,  4, 29, 11,  2, 29,
       19, 25]), 1: array([34,  6, 27, 27,  9, 32, 10, 36, 26, 16,  6, 18, 32,  8, 17, 22, 25,
        3, 29])}
1
strain_number:  0 20.263157894736842
strain_various:  0 13.415788441287566
strain_number:  1 20.36842105263158
strain_various:  1 10.683043752728652
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 23.894736842105264
strain_various:  0 9.380240917353351
strain_number:  1 20.36842105263158
strain_various:  1 3.4824450925627937
fd_r_threshold:  [1.05, 497.5118713079758]
slip_r:  100.34237426159515
3
strain_number:  0 28.263157894736842
strain_various:  0 8.800969475683424
strain_number:  1 20.36842105263158
strain_various:  1 4.028980886106239
fd_r_threshold:  [1.05, 497.5118713079758, 22.130131787284157]
slip_r:  104.55840061905198
4
strain_number:  0 33.526315789473685
strain_various:  0 9.074941908195287
strain_number:  1 20.42105263157895
strain_various:  1 5.029552553852754
fd_r_thres

glc__D_e_1:  -6.861257356604563
strain_number:  0 15.263157894736842
strain_various:  0 6.592197183937337
strain_number:  1 8.526315789473685
strain_various:  1 3.857575200380778
fd_r_threshold:  [1.05, 497.5118713079758, 22.130131787284157, 7.980818692552196, 10.126307690529623, 9.841168416987788, 6.617459005543478, 5.187436747422879, 4.305623300158765, 4.4124277756492685, 4.781477806791911, 3.8217636908566797]
slip_r:  4.5017458641759
13
glc__D_e_1:  -9.388663553471368
glc__D_e_1:  -6.9702045040007885
glc__D_e_1:  -4.967871804169486
glc__D_e_1:  -3.111885237044528
glc__D_e_1:  -6.932957863259532
glc__D_e_1:  -6.160735754255562
glc__D_e_1:  -6.9791797572520355
glc__D_e_1:  -5.6705774625620435
glc__D_e_1:  -3.9907301521156056
glc__D_e_1:  -3.5273968867132237
glc__D_e_1:  -4.108864412179218
glc__D_e_1:  -4.220047919717521
glc__D_e_1:  -10.885444566040256
glc__D_e_1:  -9.649889191633903
glc__D_e_1:  -1.8415782145524808
glc__D_e_1:  -2.880755005653263
glc__D_e_1:  -5.131856046613417
glc__

glc__D_e_1:  -0.48049774990618677
glc__D_e_1:  -0.9947273765379034
glc__D_e_1:  -0.487611005259327
glc__D_e_1:  -0.4881952348858003
glc__D_e_1:  -0.31014689890616665
glc__D_e_1:  -0.29423603043865265
glc__D_e_1:  -0.13979160863785856
glc__D_e_1:  -1.3842394559767683
glc__D_e_1:  -0.45721501064721193
glc__D_e_1:  -0.9714446372789287
glc__D_e_1:  -0.3183223448337231
glc__D_e_1:  -0.3609648232449001
glc__D_e_1:  -0.4813623085727574
glc__D_e_1:  -0.37486548298829403
glc__D_e_1:  -0.8890951096200106
glc__D_e_1:  -0.6728544385596285
glc__D_e_1:  -0.4273842457376844
glc__D_e_1:  -0.013269085185021456
glc__D_e_1:  -0.9478216487337714
glc__D_e_1:  -1.63129902467188
glc__D_e_1:  -0.49540507928266053
glc__D_e_1:  -0.34096065748186655
glc__D_e_1:  -0.3593897973638631
glc__D_e_1:  -0.8736945370244262
strain_number:  0 2.4210526315789473
strain_various:  0 0.5907880084379907
strain_number:  1 1.4210526315789473
strain_various:  1 0.6740130776245103
fd_r_threshold:  [1.05, 497.5118713079758, 22.13013

strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 497.5118713079758, 22.130131787284157, 7.980818692552196, 10.126307690529623, 9.841168416987788, 6.617459005543478, 5.187436747422879, 4.305623300158765, 4.4124277756492685, 4.781477806791911, 3.8217636908566797, 3.7583527437870843, 4.612040521163284, 3.4635328865966986, 3.7195825459813556, 4.688951247165533, 5.172499999999999, 5.611111111111111, 4.333333333333334, 6.145833333333334, 7.611111111111111, 0.75, 3.25, 4.0, 4.0, 0.0, 2.0]
slip_r:  2.65
29
strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 497.5118713079758, 22.130131787284157, 7.980818692552196, 10.126307690529623, 9.841168416987788, 6.617459005543478, 5.187436747422879, 4.305623300158765, 4.4124277756492685, 4.781477806791911, 3.8217636908566797, 3.7583527437870843, 4.612040521163284, 3.4635328

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 3, 15, 13, 28, 37, 27, 14, 11, 33, 33, 24, 21, 35, 23, 13, 17, 10,
       11, 30]), 1: array([38, 22,  9, 36, 13, 28, 24, 21, 23,  6, 27, 17, 16,  6,  1, 25, 12,
       20,  3])}
1
strain_number:  0 24.736842105263158
strain_various:  0 11.661191169918492
strain_number:  1 18.36842105263158
strain_various:  1 10.433804000841585
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 29.31578947368421
strain_various:  0 9.86782454337513
strain_number:  1 18.36842105263158
strain_various:  1 6.384277779834075
fd_r_threshold:  [1.05, 65.82645606238145]
slip_r:  14.00529121247629
3
strain_number:  0 34.78947368421053
strain_various:  0 9.939707437018356
strain_number:  1 18.36842105263158
strain_various:  1 5.886742699675973
fd_r_threshold:  [1.05, 65.82645606238145, 5.790701097176679]
slip_r:  14.953431431911628
4
strain_number:  0 41.36842105263158
strain_various:  0 10.32732974259021
strain_number:  1 18.42105263157895
strain_various:  1 6.055148218218802
fd_r_threshold:  

glc__D_e_1:  -7.9864549538545475
glc__D_e_1:  -5.437197665384838
glc__D_e_1:  -3.482071881316641
glc__D_e_1:  -3.7304829262672463
glc__D_e_1:  -7.284214064466256
glc__D_e_1:  -9.756376147485899
glc__D_e_1:  -3.3258728074202555
glc__D_e_1:  -3.009122400382055
glc__D_e_1:  -10.93339445280425
glc__D_e_1:  -15.212352776394074
glc__D_e_1:  -6.158049603827722
glc__D_e_1:  -2.0221382170044566
glc__D_e_1:  -7.979240194763391
glc__D_e_1:  -18.437300398711677
glc__D_e_1:  -6.399412188813695
glc__D_e_1:  -1.7174234981953294
glc__D_e_1:  -9.641695550617525
glc__D_e_1:  -16.89215016399191
glc__D_e_1:  -5.0087063758947235
strain_number:  0 14.631578947368421
strain_various:  0 6.225672876893156
strain_number:  1 6.894736842105263
strain_various:  1 3.2100948804161904
fd_r_threshold:  [1.05, 65.82645606238145, 5.790701097176679, 5.77894496786909, 6.63911126761423, 6.85832988781793, 6.034990708267384, 4.639706043944413, 4.698968095912722, 4.451739998919979, 4.393336408166577, 3.90721198687627]
slip_r:

glc__D_e_1:  -0.19497947569997987
glc__D_e_1:  -0.7092091023316964
glc__D_e_1:  -0.34998046593441523
glc__D_e_1:  -0.16449682367841856
glc__D_e_1:  -0.2659008102549518
glc__D_e_1:  -0.11145638845415773
glc__D_e_1:  -1.0264697522701975
glc__D_e_1:  -0.33767534665091037
glc__D_e_1:  -0.1832309248501165
glc__D_e_1:  -0.2418144537764515
glc__D_e_1:  -0.48767611319699733
glc__D_e_1:  -0.053730856442939956
glc__D_e_1:  -0.4574084277298176
glc__D_e_1:  -0.9724402537685907
glc__D_e_1:  -0.8179958319677967
glc__D_e_1:  -1.1208404233653502
glc__D_e_1:  -0.18139500531554864
glc__D_e_1:  -0.27635684116517334
glc__D_e_1:  -0.12191241936437924
glc__D_e_1:  -0.1487575548373692
glc__D_e_1:  -1.2654230537520625
glc__D_e_1:  -0.4423045835187579
strain_number:  0 2.473684210526316
strain_various:  0 1.229823310057676
strain_number:  1 1.0526315789473684
strain_various:  1 0.6046908048987398
fd_r_threshold:  [1.05, 65.82645606238145, 5.790701097176679, 5.77894496786909, 6.63911126761423, 6.85832988781793,

glc__D_e_1:  -0.04059821904989702
glc__D_e_1:  -0.0777847824590352
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 65.82645606238145, 5.790701097176679, 5.77894496786909, 6.63911126761423, 6.85832988781793, 6.034990708267384, 4.639706043944413, 4.698968095912722, 4.451739998919979, 4.393336408166577, 3.90721198687627, 4.163854273998308, 3.5941284647830187, 3.7798036519689675, 7.0287074829931955, 4.617901234567901, 5.196180555555555, 7.103741496598639, 2.4722222222222223, 7.222222222222222, 4.0, 6.5, 2.25, 3.0, 5.0, 0.0, 2.0]
slip_r:  2.45
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 65.82645606238145, 5.790701097176679, 5.77894496786909, 6.63911126761423, 6.85832988781793, 6.034990708267384, 4.639706043944413, 4.698968095912722, 4.451739998919979, 4.393336408166577, 3.90721198687627, 4.1

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 6, 28, 35,  6, 11, 13, 32, 34, 33, 25,  1, 27,  4, 30,  2, 26,  4,
       34, 29]), 1: array([25, 13, 21, 17, 39, 28,  8, 30, 11, 29,  5, 33, 28, 21, 15, 39, 15,
       24, 25])}
1
strain_number:  0 23.57894736842105
strain_various:  0 15.079842905344256
strain_number:  1 22.63157894736842
strain_various:  1 9.788341756857005
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 27.894736842105264
strain_various:  0 10.093743433914659
strain_number:  1 22.68421052631579
strain_various:  1 4.757848159995588
fd_r_threshold:  [1.05, 473.4238697698207]
slip_r:  95.52477395396414
3
strain_number:  0 33.05263157894737
strain_various:  0 7.301576500358105
strain_number:  1 22.68421052631579
strain_various:  1 4.142877611400932
fd_r_threshold:  [1.05, 473.4238697698207, 11.206701611224577]
slip_r:  97.55611427620906
4
strain_number:  0 39.1578947368421
strain_various:  0 7.828847854298982
strain_number:  1 22.736842105263158
strain_various:  1 4.597964140598222
fd_r_threshold: 

glc__D_e_1:  -7.746364155078025
glc__D_e_1:  -4.2055524566428515
glc__D_e_1:  -3.333150926849523
glc__D_e_1:  -5.495772531749978
glc__D_e_1:  -4.209320796114291
glc__D_e_1:  -3.6519167865801956
glc__D_e_1:  -3.763100294118498
glc__D_e_1:  -9.93234368963253
glc__D_e_1:  -8.851232737026972
glc__D_e_1:  -2.1227747197655114
glc__D_e_1:  -3.653744029532109
glc__D_e_1:  -8.512488059933013
glc__D_e_1:  -8.25449557756076
glc__D_e_1:  -7.59726451396055
glc__D_e_1:  -8.25583229393382
glc__D_e_1:  -8.73712349237601
glc__D_e_1:  -7.810456961571247
glc__D_e_1:  -9.398277835740055
glc__D_e_1:  -11.040430653044954
glc__D_e_1:  -5.695648467662361
glc__D_e_1:  -5.746544828891695
glc__D_e_1:  -8.341733791636878
glc__D_e_1:  -8.016716534278517
glc__D_e_1:  -3.9265678139337075
glc__D_e_1:  -5.469256693828857
glc__D_e_1:  -5.656350489535036
glc__D_e_1:  -6.259326515739154
glc__D_e_1:  -5.919388101224459
glc__D_e_1:  -5.147165992220488
strain_number:  0 12.31578947368421
strain_various:  0 3.245282107256419

glc__D_e_1:  -1.6077920149758456
glc__D_e_1:  -1.2989031713742576
glc__D_e_1:  -0.6886420274095393
glc__D_e_1:  -1.2028716540412558
glc__D_e_1:  -0.32594739783001403
glc__D_e_1:  -0.18006408691551323
glc__D_e_1:  -0.6942937135472298
strain_number:  0 3.1052631578947367
strain_various:  0 0.9116056881941459
strain_number:  1 1.736842105263158
strain_various:  1 0.713929471907923
fd_r_threshold:  [1.05, 473.4238697698207, 11.206701611224577, 5.643185662603215, 11.321506093804976, 10.376060235374169, 6.554247016545352, 5.819662106021392, 6.4083371239981775, 5.5522084462722, 4.253560803312881, 6.60541136679171, 3.563856298383258, 4.166520759144594, 4.000403786629436, 4.828673469387755, 8.148951247165533]
slip_r:  4.941681112142115
18
glc__D_e_1:  -0.27762786991533783
glc__D_e_1:  -0.0020242516830557555
glc__D_e_1:  -0.3671619773736161
glc__D_e_1:  -0.7616667021322228
glc__D_e_1:  -0.4535403364902575
glc__D_e_1:  -0.2990959146894636
glc__D_e_1:  -0.1654049922062264
glc__D_e_1:  -0.088701263

glc__D_e_1:  -0.1617960938218484
glc__D_e_1:  -0.09565548216860115
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 473.4238697698207, 11.206701611224577, 5.643185662603215, 11.321506093804976, 10.376060235374169, 6.554247016545352, 5.819662106021392, 6.4083371239981775, 5.5522084462722, 4.253560803312881, 6.60541136679171, 3.563856298383258, 4.166520759144594, 4.000403786629436, 4.828673469387755, 8.148951247165533, 4.068055555555556, 4.555555555555555, 10.305555555555555, 5.5, 2.5, 2.75, 3.0, 3.0, 6.0, 3.0]
slip_r:  3.55
28
glc__D_e_1:  -0.003351700046054229
glc__D_e_1:  -0.19911513097374922
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 473.4238697698207, 11.206701611224577, 5.643185662603215, 11.321506093804976, 10.376060235374169, 6.554247016545352, 5.819662106021392, 6.4083371239981775, 5

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([18, 14, 36, 30, 24,  4, 19, 14, 33, 34, 17, 18, 35, 17, 36,  8, 15,
       21,  4]), 1: array([19,  9, 18, 18, 22, 18, 30,  8, 27, 28, 25, 17, 21, 16, 15,  0, 25,
       17,  4])}
1
strain_number:  0 24.57894736842105
strain_various:  0 12.406771451080058
strain_number:  1 17.789473684210527
strain_various:  1 7.911210881889607
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 29.157894736842106
strain_various:  0 8.324113089438525
strain_number:  1 17.789473684210527
strain_various:  1 5.2473446655219025
fd_r_threshold:  [1.05, 139.27222297531122]
slip_r:  28.694444595062244
3
strain_number:  0 34.68421052631579
strain_various:  0 9.130934301528873
strain_number:  1 17.789473684210527
strain_various:  1 4.741518159812719
fd_r_threshold:  [1.05, 139.27222297531122, 6.8599872638229416]
slip_r:  29.856442047826835
4
strain_number:  0 41.26315789473684
strain_various:  0 11.387167779026568
strain_number:  1 17.842105263157894
strain_various:  1 5.7332195225878815
fd_r_t

glc__D_e_1:  -4.7548895008554615
glc__D_e_1:  -2.1518566382933493
glc__D_e_1:  -2.2074483920625005
glc__D_e_1:  -11.781896830150181
glc__D_e_1:  -6.534297165148765
glc__D_e_1:  -1.4575529555513091
glc__D_e_1:  -2.988522265317907
glc__D_e_1:  -8.5034886279331
glc__D_e_1:  -4.902125903398294
glc__D_e_1:  -3.922332394434741
glc__D_e_1:  -10.918611163294457
glc__D_e_1:  -16.358445679489616
glc__D_e_1:  -5.14367593982494
glc__D_e_1:  -3.0272507187681734
glc__D_e_1:  -10.951522771190369
glc__D_e_1:  -30.53989654123918
glc__D_e_1:  -9.911038424177228
strain_number:  0 15.578947368421053
strain_various:  0 9.05140750598133
strain_number:  1 6.842105263157895
strain_various:  1 2.4118819447136
fd_r_threshold:  [1.05, 139.27222297531122, 6.8599872638229416, 9.800774729802816, 10.326032336835537, 11.962697222230211, 6.783151919621866, 4.91858739902623, 4.837712277539807, 5.086225162910973, 5.10452447026376, 4.331300256298734]
slip_r:  4.8556699132079
13
glc__D_e_1:  -8.943683532717088
glc__D_e_1:

glc__D_e_1:  -0.31006560648161985
glc__D_e_1:  -0.3350975384177026
glc__D_e_1:  -0.40516539599583234
glc__D_e_1:  -0.9193950226275489
glc__D_e_1:  -0.8235205847793865
glc__D_e_1:  -0.6018279224740781
glc__D_e_1:  -0.4798607269193105
glc__D_e_1:  -0.6716197117440972
glc__D_e_1:  -1.4004699775014904
glc__D_e_1:  -0.4428750945964648
glc__D_e_1:  -0.6046892887544396
glc__D_e_1:  -1.7144517619815935
glc__D_e_1:  -0.06821482151498426
strain_number:  0 3.0526315789473686
strain_various:  0 1.190916684103659
strain_number:  1 0.7894736842105263
strain_various:  1 0.6942582083301536
fd_r_threshold:  [1.05, 139.27222297531122, 6.8599872638229416, 9.800774729802816, 10.326032336835537, 11.962697222230211, 6.783151919621866, 4.91858739902623, 4.837712277539807, 5.086225162910973, 5.10452447026376, 4.331300256298734, 4.832433798848379, 4.646169541545207, 6.74387963913376, 5.774883471907282, 4.742874149659864, 5.906944444444444]
slip_r:  5.562950249338112
19
glc__D_e_1:  -0.011297940621940894
glc__D

strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 139.27222297531122, 6.8599872638229416, 9.800774729802816, 10.326032336835537, 11.962697222230211, 6.783151919621866, 4.91858739902623, 4.837712277539807, 5.086225162910973, 5.10452447026376, 4.331300256298734, 4.832433798848379, 4.646169541545207, 6.74387963913376, 5.774883471907282, 4.742874149659864, 5.906944444444444, 6.093333333333334, 2.673611111111111, 7.611111111111111, 3.611111111111111, 2.75, 4.25, 5.0, 4.0, 2.0, 1.0, 1.0]
slip_r:  2.6
30
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 139.27222297531122, 6.8599872638229416, 9.800774729802816, 10.326032336835537, 11.962697222230211, 6.783151919621866, 4.91858739902623, 4.837712277539807, 5.086225162910973, 5.10452447026376, 4.331300256298734, 4.832433798848379, 4.646169541545207, 6.74387963913376, 5.774883471907282, 4.742874149659864, 5.906944444444

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([34,  2, 11, 21, 15, 28, 23, 24,  5, 19, 34, 32, 33,  2, 26, 30, 38,
       35, 34]), 1: array([18, 32, 30, 22, 37, 14, 15,  0, 25, 32,  1, 17, 37,  4, 16, 15, 31,
       20,  0])}
1
strain_number:  0 27.736842105263158
strain_various:  0 13.458873438673354
strain_number:  1 19.57894736842105
strain_various:  1 12.15823649553834
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.78947368421053
strain_various:  0 11.404426413820856
strain_number:  1 19.63157894736842
strain_various:  1 7.042805409298272
fd_r_threshold:  [1.05, 599.1608790307832]
slip_r:  120.67217580615666
3
strain_number:  0 38.94736842105263
strain_various:  0 12.08946337327425
strain_number:  1 19.68421052631579
strain_various:  1 5.2621051578736795
fd_r_threshold:  [1.05, 599.1608790307832, 10.702038551184046]
slip_r:  122.60258351639345
4
strain_number:  0 46.36842105263158
strain_various:  0 13.07553169419403
strain_number:  1 19.736842105263158
strain_various:  1 5.229894889385122
fd_r_thresho

glc__D_e_1:  -11.392038831583168
glc__D_e_1:  -12.162505602474347
glc__D_e_1:  -3.6101846245378346
glc__D_e_1:  -2.245990576078692
glc__D_e_1:  -3.482479648764466
glc__D_e_1:  -3.6878204317945946
glc__D_e_1:  -4.786906727895725
glc__D_e_1:  -2.930920160770767
glc__D_e_1:  -12.474635814948945
glc__D_e_1:  -9.078613873444267
glc__D_e_1:  -2.567924671243785
glc__D_e_1:  -5.0824790183420125
glc__D_e_1:  -13.30393958995364
glc__D_e_1:  -9.907917648448962
glc__D_e_1:  -3.918647063304122
glc__D_e_1:  -5.50520812683987
glc__D_e_1:  -6.035729867625367
glc__D_e_1:  -2.5888115648913543
glc__D_e_1:  -3.7809329849697892
glc__D_e_1:  -6.842871604502984
glc__D_e_1:  -11.165514903126397
glc__D_e_1:  -6.5865892865574915
glc__D_e_1:  -4.100766835137003
glc__D_e_1:  -7.654497973336013
glc__D_e_1:  -6.354675149033708
glc__D_e_1:  -2.9077568462996957
glc__D_e_1:  -3.589698798538017
glc__D_e_1:  -5.176259862073765
glc__D_e_1:  -10.002484748006939
glc__D_e_1:  -5.578003553238828
glc__D_e_1:  -0.9824114840122

glc__D_e_1:  -1.1994770049889394
glc__D_e_1:  -1.1720844840181903
glc__D_e_1:  -1.673170741462575
glc__D_e_1:  -0.02693380099596565
glc__D_e_1:  -0.09383429223600781
glc__D_e_1:  -0.6210693236228346
glc__D_e_1:  -0.46662490182204064
glc__D_e_1:  -1.0951143675130248
glc__D_e_1:  -1.6758327465865719
glc__D_e_1:  -0.48191703187270396
strain_number:  0 3.210526315789474
strain_various:  0 1.3210421471590668
strain_number:  1 1.2105263157894737
strain_various:  1 0.8321783316232577
fd_r_threshold:  [1.05, 599.1608790307832, 10.702038551184046, 7.564392972067566, 9.390737617123035, 11.965203789437082, 5.901886258007818, 5.0040364614411175, 4.889497232733089, 5.047411755385496, 5.8055962397138865, 4.32870360282489, 4.584358906757783, 3.7131531390487016, 4.325225200885375, 5.460060626102292, 5.245141723356009]
slip_r:  4.665587919230033
18
glc__D_e_1:  -1.0614059509784126
glc__D_e_1:  -2.24430962604264
glc__D_e_1:  -0.17563528450457988
glc__D_e_1:  -0.7594091996909296
glc__D_e_1:  -0.411761963

glc__D_e_1:  -0.031081271874279393
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 599.1608790307832, 10.702038551184046, 7.564392972067566, 9.390737617123035, 11.965203789437082, 5.901886258007818, 5.0040364614411175, 4.889497232733089, 5.047411755385496, 5.8055962397138865, 4.32870360282489, 4.584358906757783, 3.7131531390487016, 4.325225200885375, 5.460060626102292, 5.245141723356009, 6.300555555555556, 12.506944444444443, 3.9722222222222223, 0.9722222222222222, 4.75, 3.75, 4.0, 5.0, 3.25, 1.0]
slip_r:  3.4
28
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 599.1608790307832, 10.702038551184046, 7.564392972067566, 9.390737617123035, 11.965203789437082, 5.901886258007818, 5.0040364614411175, 4.889497232733089, 5.047411755385496, 5.8055962397138865, 4.32870360282489, 4.584358906757783, 3.71315

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([10, 17,  0, 29, 16, 34, 38, 27, 27, 31, 29, 32, 27,  2, 36, 16,  8,
       30, 18]), 1: array([20, 32, 35, 10, 17, 25, 32, 28,  0, 33, 13,  5,  5, 28,  6, 12, 33,
       35, 10])}
1
strain_number:  0 26.57894736842105
strain_various:  0 13.251874900015128
strain_number:  1 20.263157894736842
strain_various:  1 12.056193452081656
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 31.57894736842105
strain_various:  0 11.99261083673778
strain_number:  1 20.263157894736842
strain_various:  1 5.901312122277802
fd_r_threshold:  [1.05, 494.83338575057985]
slip_r:  99.80667715011597
3
strain_number:  0 37.526315789473685
strain_various:  0 14.496345895477578
strain_number:  1 20.31578947368421
strain_various:  1 5.419775015518316
fd_r_threshold:  [1.05, 494.83338575057985, 12.040588876654098]
slip_r:  102.00479492544679
4
strain_number:  0 44.63157894736842
strain_various:  0 14.064552404859192
strain_number:  1 20.42105263157895
strain_various:  1 5.941077250126171
fd_r_thre

glc__D_e_1:  -8.789066073989229
glc__D_e_1:  -8.955841335296682
glc__D_e_1:  -4.101692388101778
glc__D_e_1:  -3.6383591226993963
glc__D_e_1:  -3.7493102682313038
glc__D_e_1:  -3.368701257103791
glc__D_e_1:  -10.232205772562166
glc__D_e_1:  -9.151094819956608
glc__D_e_1:  -1.4414128973984455
glc__D_e_1:  -3.955967244496673
glc__D_e_1:  -8.961292500840981
glc__D_e_1:  -4.53681130607287
glc__D_e_1:  -2.8664938315268724
glc__D_e_1:  -13.57754902039214
glc__D_e_1:  -5.346364318059095
glc__D_e_1:  -0.5134131662026569
glc__D_e_1:  -3.0279675133008843
glc__D_e_1:  -17.775633791271897
glc__D_e_1:  -8.566886196904752
glc__D_e_1:  -0.47989328828710964
glc__D_e_1:  -0.5354850420562611
glc__D_e_1:  -11.421081151504152
glc__D_e_1:  -6.1734814865027365
glc__D_e_1:  -3.377793222479789
glc__D_e_1:  -4.472561767349722
glc__D_e_1:  -13.467546167217565
glc__D_e_1:  -7.242383610182049
glc__D_e_1:  -4.6783206140850915
glc__D_e_1:  -6.264881677620839
glc__D_e_1:  -7.865990920699273
glc__D_e_1:  -5.6019762930

glc__D_e_1:  -0.652092462730862
glc__D_e_1:  -0.618768099036854
glc__D_e_1:  -1.5988371012955223
glc__D_e_1:  -0.08774172022220683
glc__D_e_1:  -0.6128537273966321
glc__D_e_1:  -1.1270833540283487
glc__D_e_1:  -1.4074142642997458
glc__D_e_1:  -0.9712134994030817
glc__D_e_1:  -1.2574238105773379
glc__D_e_1:  -1.1029793887765438
strain_number:  0 2.8947368421052633
strain_various:  0 0.851916529275718
strain_number:  1 1.5263157894736843
strain_various:  1 0.5954583420518295
fd_r_threshold:  [1.05, 494.83338575057985, 12.040588876654098, 8.013454915003715, 6.523577645865121, 7.933557071380964, 5.339694258176137, 2.4262389565528615, 4.733046223372115, 4.384784549403243, 4.7950965881161345, 3.996291460827154, 5.531958047855968, 3.6452535455565767, 4.163345760480501, 5.507840136054422, 5.582222222222221]
slip_r:  4.8861239424339376
18
glc__D_e_1:  -0.9750668266142715
glc__D_e_1:  -0.6966042516512836
glc__D_e_1:  -0.4909002417875912
glc__D_e_1:  -0.5746836837865199
glc__D_e_1:  -0.7233930470

glc__D_e_1:  -0.09321280659536335
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 494.83338575057985, 12.040588876654098, 8.013454915003715, 6.523577645865121, 7.933557071380964, 5.339694258176137, 2.4262389565528615, 4.733046223372115, 4.384784549403243, 4.7950965881161345, 3.996291460827154, 5.531958047855968, 3.6452535455565767, 4.163345760480501, 5.507840136054422, 5.582222222222221, 4.2780555555555555, 5.333333333333334, 5.833333333333334, 8.75, 1.75, 4.25, 2.25, 7.0, 0.0, 1.25]
slip_r:  2.95
28
glc__D_e_1:  -0.19163728599546304
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 494.83338575057985, 12.040588876654098, 8.013454915003715, 6.523577645865121, 7.933557071380964, 5.339694258176137, 2.4262389565528615, 4.733046223372115

strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 494.83338575057985, 12.040588876654098, 8.013454915003715, 6.523577645865121, 7.933557071380964, 5.339694258176137, 2.4262389565528615, 4.733046223372115, 4.384784549403243, 4.7950965881161345, 3.996291460827154, 5.531958047855968, 3.6452535455565767, 4.163345760480501, 5.507840136054422, 5.582222222222221, 4.2780555555555555, 5.333333333333334, 5.833333333333334, 8.75, 1.75, 4.25, 2.25, 7.0, 0.0, 1.25, 1.0, 2.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
slip_r:  0.0
The change process of microorganism distribution is:  {0: {0: array([10, 17,  0, 29, 16, 34, 38, 27, 27, 31, 29, 32, 27,  2, 36, 16,  8,
       30, 18]), 1: array([20, 32, 35, 10, 17, 25, 32, 28,  0, 33, 13,  5,  5, 28,  6, 12, 33,
       35, 10])}, 1: {0: array([12, 20,  0, 34, 19, 40, 45, 32, 32, 37, 34, 38, 32,  2, 43, 19,  9,
       36, 21]), 1: array([20, 3

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([13, 12, 39, 37, 25,  5,  9, 37, 32, 15,  7, 37, 38,  9, 28, 19, 25,
       20, 14]), 1: array([13, 39, 10, 12, 32, 18, 37, 36,  8,  7, 10,  8, 39, 14, 20, 28, 34,
       34, 25])}
1
strain_number:  0 26.157894736842106
strain_various:  0 13.846007054647224
strain_number:  1 22.68421052631579
strain_various:  1 12.126754378033404
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 31.0
strain_various:  0 8.397994748121345
strain_number:  1 22.842105263157894
strain_various:  1 7.842281877205846
fd_r_threshold:  [1.05, 31.287779027613833]
slip_r:  7.0975558055227665
3
strain_number:  0 36.78947368421053
strain_various:  0 7.208028764534732
strain_number:  1 23.0
strain_various:  1 6.680687011630496
fd_r_threshold:  [1.05, 31.287779027613833, 6.747229913422746]
slip_r:  8.237001788207317
4
strain_number:  0 43.78947368421053
strain_various:  0 8.313457367858586
strain_number:  1 23.105263157894736
strain_various:  1 5.857022207681497
fd_r_threshold:  [1.05, 31.28777902761

glc__D_e_1:  -7.400153785685537
glc__D_e_1:  -3.7987910611507303
glc__D_e_1:  -2.125095821484816
glc__D_e_1:  -4.1478576499172295
glc__D_e_1:  -10.123497863516755
glc__D_e_1:  -7.705038814046176
glc__D_e_1:  -2.696856303339899
glc__D_e_1:  -5.211410650438126
glc__D_e_1:  -13.396036396832706
glc__D_e_1:  -8.508221936662213
glc__D_e_1:  -1.7235654821228692
glc__D_e_1:  -2.2709497545578357
glc__D_e_1:  -10.825311129620479
glc__D_e_1:  -7.583733609916595
glc__D_e_1:  -0.2882043077937908
glc__D_e_1:  -11.832749520702597
glc__D_e_1:  -8.591172000998714
glc__D_e_1:  -0.6511514318099412
glc__D_e_1:  -3.1657057789081686
glc__D_e_1:  -11.820166637352338
glc__D_e_1:  -7.241241020783433
glc__D_e_1:  -0.6978752766625822
glc__D_e_1:  -1.2452595490975487
glc__D_e_1:  -11.102899343611346
glc__D_e_1:  -7.192647775474952
glc__D_e_1:  -1.6379085322741747
glc__D_e_1:  -3.660670360706588
glc__D_e_1:  -9.457321258966108
glc__D_e_1:  -8.37621030636055
glc__D_e_1:  -1.071869701504406
glc__D_e_1:  -1.127461455

glc__D_e_1:  -1.6358519081474812
glc__D_e_1:  -1.9956371129784038
glc__D_e_1:  -0.6075264525681741
glc__D_e_1:  -1.2530243634260199
glc__D_e_1:  -1.7672539900577364
strain_number:  0 2.4210526315789473
strain_various:  0 0.8153649149910351
strain_number:  1 1.8421052631578947
strain_various:  1 0.7443229275647869
fd_r_threshold:  [1.05, 31.287779027613833, 6.747229913422746, 6.605017552775387, 7.01582563048736, 6.6873167721958335, 4.364511611738736, 3.76279115767073, 5.030687117885795, 3.9553020713117104, 3.380930681139672, 3.6380794281479014, 3.2678735973573287, 5.134678448694224, 4.073197278911565, 3.7296541950113387, 3.9447222222222225]
slip_r:  4.030025148439336
18
glc__D_e_1:  -0.4417656556308114
glc__D_e_1:  -0.6398326950076947
glc__D_e_1:  -0.7779331820960631
glc__D_e_1:  -0.6234887602952691
glc__D_e_1:  -0.44538232087971164
glc__D_e_1:  -0.530416568127285
glc__D_e_1:  -0.9996732825240287
glc__D_e_1:  -0.09361440686391376
glc__D_e_1:  -0.08705460003299981
glc__D_e_1:  -1.0348631

strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 31.287779027613833, 6.747229913422746, 6.605017552775387, 7.01582563048736, 6.6873167721958335, 4.364511611738736, 3.76279115767073, 5.030687117885795, 3.9553020713117104, 3.380930681139672, 3.6380794281479014, 3.2678735973573287, 5.134678448694224, 4.073197278911565, 3.7296541950113387, 3.9447222222222225, 9.131944444444443, 6.722222222222222, 4.361111111111111, 2.75, 6.611111111111111, 4.5, 1.0, 6.25, 4.0, 3.0, 0.0]
slip_r:  2.85
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 31.287779027613833, 6.747229913422746, 6.605017552775387, 7.01582563048736, 6.6873167721958335, 4.364511611738736, 3.76279115767073, 5.030687117885795, 3.9553020713117104, 3.380930681139672, 3.6380794281479014, 3.2678735973573287, 5.134678448694224, 4.073197278911565, 

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 1, 13, 33, 21, 33, 11, 15, 39, 25, 22, 33, 20, 21, 33, 34,  3, 15,
       38, 37]), 1: array([ 9, 38, 14, 18, 39,  1, 39,  1,  5, 11, 12, 17, 14, 30,  2, 18, 13,
       33, 19])}
1
strain_number:  0 27.842105263157894
strain_various:  0 13.54626830686412
strain_number:  1 17.789473684210527
strain_various:  1 12.693052170664163
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 33.05263157894737
strain_various:  0 11.278641595187205
strain_number:  1 17.789473684210527
strain_various:  1 6.606049439466519
fd_r_threshold:  [1.05, 465.05888767948124]
slip_r:  93.85177753589625
3
strain_number:  0 39.26315789473684
strain_various:  0 10.009690595476691
strain_number:  1 17.789473684210527
strain_various:  1 5.366356672223531
fd_r_threshold:  [1.05, 465.05888767948124, 7.589269826140493]
slip_r:  95.15963150112434
4
strain_number:  0 46.68421052631579
strain_various:  0 13.588124194073787
strain_number:  1 17.789473684210527
strain_various:  1 4.059802268563556
fd_r_thre

glc__D_e_1:  -3.4247762445274086
glc__D_e_1:  -2.5523747147340803
glc__D_e_1:  -4.65347899993732
glc__D_e_1:  -2.698353215869123
glc__D_e_1:  -2.7855651622313715
glc__D_e_1:  -7.759082102658676
glc__D_e_1:  -13.587341573264336
glc__D_e_1:  -4.6874828224987795
glc__D_e_1:  -0.4628659649161402
glc__D_e_1:  -14.703847893730256
glc__D_e_1:  -6.47266319139721
glc__D_e_1:  -1.1136313273399052
glc__D_e_1:  -3.6281856744381327
glc__D_e_1:  -13.658036781818112
glc__D_e_1:  -4.758178031052555
glc__D_e_1:  -1.6106313160172787
glc__D_e_1:  -6.584148256444582
glc__D_e_1:  -7.649427041194315
glc__D_e_1:  -3.379390268226998
glc__D_e_1:  -1.8817806733864022
glc__D_e_1:  -5.3799200578162605
glc__D_e_1:  -11.51773116136868
glc__D_e_1:  -6.270131496367264
glc__D_e_1:  -3.6706502613374403
glc__D_e_1:  -5.257211324873188
glc__D_e_1:  -6.618119716772883
glc__D_e_1:  -5.845897607768913
glc__D_e_1:  -1.4292433848349257
glc__D_e_1:  -2.9602126946015233
glc__D_e_1:  -2.680471911674576
glc__D_e_1:  -4.3776052133

glc__D_e_1:  -0.6432639907205018
glc__D_e_1:  -0.7288318135968535
glc__D_e_1:  -0.5089131625307892
glc__D_e_1:  -0.8356477671918721
glc__D_e_1:  -0.6812033453910781
glc__D_e_1:  -0.03604686834318027
glc__D_e_1:  -0.4588225068892813
glc__D_e_1:  -0.3043780850884872
strain_number:  0 3.1578947368421053
strain_various:  0 0.9326339550878605
strain_number:  1 1.105263157894737
strain_various:  1 0.5520046569316587
fd_r_threshold:  [1.05, 465.05888767948124, 7.589269826140493, 7.658546159867577, 6.719125766225914, 7.047610507831704, 5.241378593126685, 4.681768186636156, 5.241603424178289, 4.334362423241037, 4.419935424859957, 4.684225394768888, 3.4213370693991507, 3.725328882900687, 3.5379072072616036, 4.109410430839002, 5.589444444444444]
slip_r:  4.076685606968978
18
glc__D_e_1:  -0.6480969233998364
glc__D_e_1:  -0.06653527950689808
glc__D_e_1:  -0.757880259094021
glc__D_e_1:  -0.154081255201866
glc__D_e_1:  -0.2655967488771587
glc__D_e_1:  -0.8894575929867345
glc__D_e_1:  -0.830395754939

strain_number:  0 0.3157894736842105
strain_various:  0 0.46482951928041294
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 465.05888767948124, 7.589269826140493, 7.658546159867577, 6.719125766225914, 7.047610507831704, 5.241378593126685, 4.681768186636156, 5.241603424178289, 4.334362423241037, 4.419935424859957, 4.684225394768888, 3.4213370693991507, 3.725328882900687, 3.5379072072616036, 4.109410430839002, 5.589444444444444, 5.498333333333333, 3.416666666666667, 9.41, 2.7222222222222223, 4.75, 2.25, 4.0, 5.0, 1.0, 0.0, 0.0]
slip_r:  2.0
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 465.05888767948124, 7.589269826140493, 7.658546159867577, 6.719125766225914, 7.047610507831704, 5.241378593126685, 4.681768186636156, 5.241603424178289, 4.334362423241037, 4.419935424859957, 4.684225394768888, 3.4213370693991507, 3.725328882900687, 3.5379072072616036, 4.10941043083900

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 6, 26,  8, 39, 11,  3, 26,  7, 22, 26, 12, 24,  0, 12,  9,  2,  4,
       35, 35]), 1: array([ 2, 12,  8, 13, 21,  8, 35, 18, 16,  4,  4, 29, 32, 15,  6, 26, 12,
       29, 38])}
1
strain_number:  0 19.0
strain_various:  0 14.531272990926917
strain_number:  1 17.42105263157895
strain_various:  1 11.240508461404854
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 22.42105263157895
strain_various:  0 10.584575617247483
strain_number:  1 17.473684210526315
strain_various:  1 7.365412919804591
fd_r_threshold:  [1.05, 128.97191397794262]
slip_r:  26.63438279558852
3
strain_number:  0 26.42105263157895
strain_various:  0 8.8989401466732
strain_number:  1 17.526315789473685
strain_various:  1 5.81620622464897
fd_r_threshold:  [1.05, 128.97191397794262, 13.966216366095667]
slip_r:  29.217626068807657
4
strain_number:  0 31.210526315789473
strain_various:  0 8.075818012133318
strain_number:  1 17.57894736842105
strain_various:  1 6.002769443950843
fd_r_threshold:  [1.05, 12

glc__D_e_1:  -16.495717228510877
glc__D_e_1:  -1.937577246683647
glc__D_e_1:  -1.0762342599105426
glc__D_e_1:  -4.082581125674586
glc__D_e_1:  -10.25077227485038
glc__D_e_1:  -5.1576170316497585
glc__D_e_1:  -2.8139426649911803
glc__D_e_1:  -11.23000723607919
glc__D_e_1:  -15.310532948149225
glc__D_e_1:  -4.918881678717852
glc__D_e_1:  -1.531965375653166
glc__D_e_1:  -2.571142166753948
glc__D_e_1:  -11.501110876737073
glc__D_e_1:  -6.253511211735657
glc__D_e_1:  -3.971394492714003
glc__D_e_1:  -6.049748074915567
glc__D_e_1:  -7.459686250417117
glc__D_e_1:  -3.1896494774498
glc__D_e_1:  -6.2763135322996515
glc__D_e_1:  -12.289007263827738
glc__D_e_1:  -11.303488950112481
glc__D_e_1:  -6.724563333543576
glc__D_e_1:  -8.094383955191274
glc__D_e_1:  -10.720121809827804
glc__D_e_1:  -8.543819128539733
glc__D_e_1:  -6.279804500869948
glc__D_e_1:  -5.420665006726379
glc__D_e_1:  -4.548263476933051
glc__D_e_1:  -7.226160463385659
glc__D_e_1:  -6.453938354381689
glc__D_e_1:  -3.67966841541825
g

glc__D_e_1:  -0.6386768341377556
glc__D_e_1:  -1.744312542128231
glc__D_e_1:  -1.2932326021819205
glc__D_e_1:  -0.47011413194861584
glc__D_e_1:  -0.08796459594488182
glc__D_e_1:  -1.3005432749074837
glc__D_e_1:  -1.8147729015392002
glc__D_e_1:  -1.4460879124774924
glc__D_e_1:  -0.5180946289150132
glc__D_e_1:  -0.05341095855307709
glc__D_e_1:  -0.5676405851847939
glc__D_e_1:  -0.8372885178051561
glc__D_e_1:  -0.7993934579078019
glc__D_e_1:  -0.847869226666909
glc__D_e_1:  -1.6171668006789066
glc__D_e_1:  -0.639603908644808
glc__D_e_1:  -0.024368579334870244
glc__D_e_1:  -0.6129134331455348
strain_number:  0 3.0
strain_various:  0 1.025978352085154
strain_number:  1 1.3157894736842106
strain_various:  1 0.7292845505553167
fd_r_threshold:  [1.05, 128.97191397794262, 13.966216366095667, 5.228184511139448, 6.986799566509116, 9.006887227440366, 6.8854045007841975, 5.7560006424174235, 4.181605637101035, 4.040651689069445, 5.123562136588582, 5.233193526070093, 4.719039893713448, 4.111200267094

strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 128.97191397794262, 13.966216366095667, 5.228184511139448, 6.986799566509116, 9.006887227440366, 6.8854045007841975, 5.7560006424174235, 4.181605637101035, 4.040651689069445, 5.123562136588582, 5.233193526070093, 4.719039893713448, 4.111200267094913, 3.980044652727814, 3.770048822861011, 5.017013101536911, 4.81718820861678, 4.260555555555555, 6.506944444444445, 6.083333333333333, 4.611111111111111, 3.75, 0.5, 4.0, 12.0, 0.0, 0.0]
slip_r:  3.3
29
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 128.97191397794262, 13.966216366095667, 5.228184511139448, 6.986799566509116, 9.006887227440366, 6.8854045007841975, 5.7560006424174235, 4.181605637101035, 4.040651689069445, 5.123562136588582, 5.233193526070093, 4.719039893713448, 4.111200267094913, 3.98004

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([12, 26,  7, 39, 36,  0, 30,  4, 30, 34, 32,  4, 36, 13, 21, 15, 39,
       24, 16]), 1: array([29, 39,  6, 21, 30, 28, 22, 33, 10, 31, 16,  5, 30,  3,  0,  3, 28,
        5,  5])}
1
strain_number:  0 26.0
strain_various:  0 15.033296378372908
strain_number:  1 18.36842105263158
strain_various:  1 12.794779059879689
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.842105263157894
strain_various:  0 6.75326944519808
strain_number:  1 18.42105263157895
strain_various:  1 8.875251157150037
fd_r_threshold:  [1.05, 712.1300352304199]
slip_r:  143.26600704608398
3
strain_number:  0 36.578947368421055
strain_various:  0 8.381155487939559
strain_number:  1 18.526315789473685
strain_various:  1 8.31212444218578
fd_r_threshold:  [1.05, 712.1300352304199, 23.599739027824597]
slip_r:  147.77595485164892
4
strain_number:  0 43.578947368421055
strain_various:  0 9.831826887440519
strain_number:  1 18.63157894736842
strain_various:  1 8.079933078903363
fd_r_threshold:  [1.05, 71

glc__D_e_1:  -4.625293453368343
glc__D_e_1:  -5.720061998238276
glc__D_e_1:  -9.22357512295656
glc__D_e_1:  -7.473790121918492
glc__D_e_1:  -2.786963669615167
glc__D_e_1:  -4.317932979381764
glc__D_e_1:  -8.675606943605562
glc__D_e_1:  -5.0742442190707555
glc__D_e_1:  -1.767426911872911
glc__D_e_1:  -3.790188740305324
glc__D_e_1:  -19.29009783617585
glc__D_e_1:  -8.589557723142889
glc__D_e_1:  -3.397609019044307
glc__D_e_1:  -9.90209526923821
glc__D_e_1:  -7.605270208149688
glc__D_e_1:  -0.6605372414523281
glc__D_e_1:  -1.4038446369682474
glc__D_e_1:  -7.360946614727181
glc__D_e_1:  -23.893741123465283
glc__D_e_1:  -5.22000879047153
glc__D_e_1:  -1.9520735408305776
glc__D_e_1:  -9.876345593252774
glc__D_e_1:  -9.926085986508365
glc__D_e_1:  -2.8269085980102115
strain_number:  0 14.052631578947368
strain_various:  0 7.776304657575076
strain_number:  1 6.315789473684211
strain_various:  1 2.5966097649566335
fd_r_threshold:  [1.05, 712.1300352304199, 23.599739027824597, 10.21855344148048,

glc__D_e_1:  -0.20345591198024726
glc__D_e_1:  -0.5445321524475042
glc__D_e_1:  -0.7432202890013633
glc__D_e_1:  -0.18183174275864022
glc__D_e_1:  -0.36380630341044906
glc__D_e_1:  -0.630612608339745
glc__D_e_1:  -0.4333591799988361
glc__D_e_1:  -0.9475888066305527
glc__D_e_1:  -0.8828368619898734
glc__D_e_1:  -0.708080899581252
glc__D_e_1:  -0.553636477780458
glc__D_e_1:  -0.6688075464412346
glc__D_e_1:  -1.008179981966956
glc__D_e_1:  -0.985285447251844
glc__D_e_1:  -0.5536601705770821
glc__D_e_1:  -1.5539068393138944
glc__D_e_1:  -1.582259288099067
strain_number:  0 2.6842105263157894
strain_various:  0 1.0786263964168
strain_number:  1 0.8421052631578947
strain_various:  1 0.5860804592452655
fd_r_threshold:  [1.05, 712.1300352304199, 23.599739027824597, 10.21855344148048, 9.833502929899206, 7.472350928389501, 4.6390921146220725, 3.8689370852559444, 6.711324295057864, 4.872291843360292, 5.012349389574557, 4.389734641454662, 3.936093958257261, 5.666577478794478, 4.470640589569161, 7.

strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 712.1300352304199, 23.599739027824597, 10.21855344148048, 9.833502929899206, 7.472350928389501, 4.6390921146220725, 3.8689370852559444, 6.711324295057864, 4.872291843360292, 5.012349389574557, 4.389734641454662, 3.936093958257261, 5.666577478794478, 4.470640589569161, 7.040543324112463, 7.669873268866775, 3.2058333333333335, 6.076666666666666, 5.284722222222222, 5.722222222222222, 1.75, 4.75, 6.0, 2.0, 4.0, 2.0, 0.0]
slip_r:  2.8
29
glc__D_e_1:  -0.06350229534197482
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 712.1300352304199, 23.599739027824597, 10.21855344148048, 9.833502929899206, 7.472350928389501, 4.6390921146220725, 3.8689370852559444, 6.711324295057864, 4.872291843360292, 5.012349389574557, 4.389734641454662, 3.936093958257261, 5.666577478794478, 4.470640589569161, 7

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([10, 22,  1, 17, 18, 37, 16,  5, 17, 13, 22,  4, 34, 23, 29,  1, 23,
       31, 18]), 1: array([39, 20, 28, 27, 23, 21, 37,  7, 30, 21,  9,  7,  1, 15, 18, 34, 18,
       17, 30])}
1
strain_number:  0 21.105263157894736
strain_various:  0 12.29845834087862
strain_number:  1 21.42105263157895
strain_various:  1 10.609408880173467
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.842105263157894
strain_various:  0 8.008305937520099
strain_number:  1 21.526315789473685
strain_various:  1 7.652783608132964
fd_r_threshold:  [1.05, 732.2511641939678]
slip_r:  147.29023283879357
3
strain_number:  0 29.473684210526315
strain_various:  0 7.492932958550319
strain_number:  1 21.63157894736842
strain_various:  1 7.190714233727811
fd_r_threshold:  [1.05, 732.2511641939678, 9.192561285131866]
slip_r:  148.91874509581993
4
strain_number:  0 34.94736842105263
strain_various:  0 9.190805956357897
strain_number:  1 21.68421052631579
strain_various:  1 6.805977493342834
fd_r_threshol

glc__D_e_1:  -16.16618506907737
glc__D_e_1:  -7.7805559449435275
glc__D_e_1:  -3.607608492451198
glc__D_e_1:  -6.177754593318577
glc__D_e_1:  -15.413614241163637
glc__D_e_1:  -9.034007262327329
glc__D_e_1:  -1.167618224134695
glc__D_e_1:  -1.2232099779038466
glc__D_e_1:  -11.18217324422619
glc__D_e_1:  -7.271921676089796
glc__D_e_1:  -4.069724646368672
glc__D_e_1:  -4.180908153906975
glc__D_e_1:  -8.452371776236692
glc__D_e_1:  -6.188357148566907
strain_number:  0 14.526315789473685
strain_various:  0 6.4348428276117575
strain_number:  1 8.263157894736842
strain_various:  1 4.127470300357401
fd_r_threshold:  [1.05, 732.2511641939678, 9.192561285131866, 9.43886318949012, 13.21349594740418, 12.656911036557736, 8.520127160389327, 6.064680694097949, 4.893464525662638, 4.6745574498754605, 4.351956608504516, 5.295425736609334]
slip_r:  5.05601700294998
13
glc__D_e_1:  -3.754535443579793
glc__D_e_1:  -9.309268614070007
glc__D_e_1:  -2.791587287758734
glc__D_e_1:  -3.3389715601937007
glc__D_e_

glc__D_e_1:  -1.530951919908605
glc__D_e_1:  -0.47538501043053116
glc__D_e_1:  -0.8198507382698677
glc__D_e_1:  -1.3340803649015842
glc__D_e_1:  -0.12300151949369842
glc__D_e_1:  -0.8452691975721089
glc__D_e_1:  -0.6991460109608583
glc__D_e_1:  -1.1152679393414333
glc__D_e_1:  -0.5336372730179768
glc__D_e_1:  -0.6884903602534016
glc__D_e_1:  -0.5340459384526076
glc__D_e_1:  -1.059899366587219
glc__D_e_1:  -0.40699034971818904
glc__D_e_1:  -0.25254592791739494
glc__D_e_1:  -0.79729278221308
glc__D_e_1:  -1.8942896781523904
glc__D_e_1:  -0.24805273768578107
glc__D_e_1:  -0.03924986457240509
glc__D_e_1:  -0.7765686764204769
glc__D_e_1:  -0.6221242546196829
glc__D_e_1:  -0.4780840989857531
glc__D_e_1:  -0.1899444840569413
glc__D_e_1:  -1.397631254804478
glc__D_e_1:  -0.30401766909171934
strain_number:  0 3.210526315789474
strain_various:  0 1.0041465278073112
strain_number:  1 0.9473684210526315
strain_various:  1 0.6046908048987398
fd_r_threshold:  [1.05, 732.2511641939678, 9.192561285131

glc__D_e_1:  -0.07499653896227487
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 732.2511641939678, 9.192561285131866, 9.43886318949012, 13.21349594740418, 12.656911036557736, 8.520127160389327, 6.064680694097949, 4.893464525662638, 4.6745574498754605, 4.351956608504516, 5.295425736609334, 4.273754812617961, 5.421121171666667, 3.2316523191722997, 6.232015778533635, 7.42721655328798, 5.947222222222223, 6.895833333333333, 5.824722222222222, 2.8125, 7.083333333333334, 2.0, 4.25, 4.25, 4.0, 4.0]
slip_r:  3.7
28
glc__D_e_1:  -0.03145630732660637
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 732.2511641939678, 9.192561285131866, 9.43886318949012, 13.21349594740418, 12.656911036557736, 8.520127160389327, 6.064680694097949, 4.893464525662638, 4.6745574498754605, 4.351956608504516, 5.295425736609334,